# Stage 1 - tile + embed (UNI2-h, Virchow2, Phikon-v2)

**Settings (right panel):** Accelerator **GPU T4 x2**, Internet **On**, Secrets `HF_TOKEN`
(and optionally `WANDB_API_KEY`) attached.

**How to run the full extraction:** use **Save Version -> Save & Run All (Commit)**, not the
interactive session: commits run in the background for up to 12 h and keep everything in
`/kaggle/working` as the notebook's output. Expect roughly 4-8 h for 203 slides (~218 GB of
JPEG streamed one slide at a time: download -> decode at 20x -> tile -> 3 encoders -> delete).

If one run is not enough, set `SHARD = "0/2"` for the first commit and `"1/2"` for a
second one, then attach both outputs to the stage-2 notebook. Completed slides are
skipped automatically on re-runs.

In [ ]:
# Clone the pinned branch of the repository and install only what Kaggle lacks.
# NOTE: no "pip -U": upgrading huggingface_hub to 2.x breaks Kaggle's transformers.
import os, subprocess, sys
REPO_DIR = "/kaggle/working/pathology-fusion-AI"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", "claude/keen-noether-926o85", "https://github.com/jkaral/pathology-fusion-AI.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
!pip install -q -r requirements-kaggle.txt
!git log -1 --format="commit %h %s"

In [ ]:
# Secrets live in Kaggle "Add-ons -> Secrets" (never in the repo or notebook).
import os
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    for name in ("HF_TOKEN", "WANDB_API_KEY"):
        try:
            os.environ[name] = s.get_secret(name)
            print(f"{name}: loaded")
        except Exception:
            print(f"{name}: not set")
except ImportError:
    print("Not on Kaggle: set HF_TOKEN / WANDB_API_KEY in the environment yourself.")
import torch
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 1. Disk check and a 2-slide smoke test (look at the QC overlays!)

In [ ]:
import shutil
print("free GB in /kaggle/working:", shutil.disk_usage("/kaggle/working").free / 1e9)
print("free GB in /tmp:", shutil.disk_usage("/tmp").free / 1e9)
os.environ["PATHFUSION_JPG_CACHE"] = "/tmp"           # JPEGs are deleted after each slide
OUT = "/kaggle/working/embeddings"
!python scripts/01_extract_embeddings.py --limit 2 --out-dir {OUT} paths.qc_dir=/kaggle/working/qc

In [ ]:
# Green = tissue outline, blue boxes = kept 224px tiles at 20x. Check pen marks / background are excluded.
from IPython.display import Image, display
import glob
for p in sorted(glob.glob("/kaggle/working/qc/*.jpg"))[:2]:
    print(p); display(Image(filename=p, width=700))

## 2. Full extraction (resumable)

In [ ]:
SHARD = None          # e.g. "0/2" and "1/2" across two commits if 12 h is not enough
shard_arg = f"--shard {SHARD}" if SHARD else ""
!python scripts/01_extract_embeddings.py {shard_arg} --out-dir {OUT} paths.qc_dir=/kaggle/working/qc

In [ ]:
!python scripts/01_extract_embeddings.py --summarize --out-dir {OUT}

## 3. Next
After the commit finishes: open the notebook's **Output** tab -> **New Dataset**, name it
`clwd-embeddings`. Attach that dataset to the stage-2 notebook. Also download
`embeddings/extraction_summary.json` and `embeddings/encoders.json` (small) for the report.